In [2]:
from pathlib import Path
import tensorflow as tf
import sys

# appending llm_components path to sys.path to easily import
axiom_utils = Path('/kaggle/input/datasets/harshit1234g/axiomlm-utils')
sys.path.append(str(axiom_utils))
import llm_components as lc

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

## Paths

In [4]:
dolly_dir = axiom_utils / 'dolly_15k'
features_path = str(dolly_dir / 'processed_features.npy')
labels_path = str(dolly_dir / 'processed_labels.npy')
tokenizer_path = str(axiom_utils / 'sp_tokenizer.model')

# I uploaded the model on kaggle, so it has a different path
model_path = Path('/kaggle/input/models/harshit1234g/axiomlm/tensorflow2/default/4/AxiomLM-33M-Base.keras')

## Loading data

In [5]:
tokenizer = lc.load_sp_tokenizer(tokenizer_path)
pad_id = tokenizer.pad_id()

In [6]:
full_ds = lc.load_sft_dataset(
    features_path,
    labels_path,
    pad_token_id= pad_id,
    batch_size= 64,
    shuffle_buffer= 10_000
)

I0000 00:00:1772727121.004761      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772727121.010839      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [7]:
batches = 0
for _ in full_ds.as_numpy_iterator():
    batches += 1

In [8]:
train_size = int(0.8 * batches)
val_size = int(0.1 * batches)
test_size = batches - train_size - val_size

In [9]:
print(f'Total batches: {batches}')
print(f'{train_size = }, {val_size = }, {test_size = }')

Total batches: 216
train_size = 172, val_size = 21, test_size = 23


In [10]:
train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size).take(val_size)
test_ds = full_ds.skip(train_size + val_size)

In [11]:
for item in train_ds.take(1):
    print(item[0][0])
    print(item[1][0])

tf.Tensor(
[    1    39  5109  1622 15984    14 15955   350  4654   262  2395 15954
 15925   982  2152   474  1943  1588  4613   368   264  3042   704 13657
    67    14    39   363  1298  2898 15984    14 15970 11901   857  6282
   772  1512 15935   264  3042  9070   368   308  2046 15935   463   376
  3249   563  3636  6775  3787 15936    14  7170  3869   338  2444   379
  1868  6282  3190 15935   264  3042   264  9043   376  5544   376  1203
  1864  3234   749   379   264  3003 15936    14  6029   737   285   264
  2145 15935  3493   285  8943  1192   292  5981  2788 15935   603   376
   474   262  4392   347   627   470 15472   638  1159 15936    14 11558
 15935   264   994  4820   287   264 11834 15922   884  1843   330   983
  1250  6231   287  5613   287 14282 10530   568   293   264  3042 15936
 15914    14 15951 15929   264  3494   297  1223   359   262  5782 15935
   355  1250   360  2108   293 10357   430   399 11834 15922 15935   501
  1366   264  9043   376  2395 15954 159

## SFT

In [12]:
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


In [13]:
with strategy.scope():
    model = tf.keras.models.load_model(model_path)
    
    # freezing the initial embedding and transformer layers
    for layer in model.layers[:6]:
        layer.trainable = False

    for layer in model.layers[6:]:
        layer.trainable = True
    
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate= 2e-5,
        weight_decay= 0.0,   # removing regularization, so that model could adapt the new behaviour easily
        clipnorm= 1.0
    )

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits= True,
        ignore_class= -100
    )

    model.compile(
        optimizer= optimizer,
        loss= loss_fn,
        metrics= [lc.Perplexity(pad_id= pad_id, ignore_index= -100)]
    )

In [14]:
for layer in model.layers:
    print(f'{layer}: {layer.trainable}')

<Embedding name=embedding, built=True>: False
<Embedding name=embedding_1, built=True>: False
<TransformerBlock name=transformer_block, built=True>: False
<TransformerBlock name=transformer_block_1, built=True>: False
<TransformerBlock name=transformer_block_2, built=True>: False
<TransformerBlock name=transformer_block_3, built=True>: False
<TransformerBlock name=transformer_block_4, built=True>: True
<TransformerBlock name=transformer_block_5, built=True>: True
<TransformerBlock name=transformer_block_6, built=True>: True
<TransformerBlock name=transformer_block_7, built=True>: True
<LayerNormalization name=layer_normalization_16, built=True>: True


In [15]:
model.summary()

Model: "gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 512, 512)          │     8,192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (512, 512)             │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_16          │ ?                      │         1,024 │
│ (LayerNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,661,952 (128.41 MB)

 Trainable params: 12,604,416 (48.08 MB)

 Non-trainable params: 21,057,536 (80.33 MB)

In [16]:
history = model.fit(
    train_ds,
    epochs= 5,
    validation_data= val_ds
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


172/172 ━━━━━━━━━━━━━━━━━━━━ 241s 1s/step - loss: 4.8452 - ppl: 2.1694 - val_loss: 3.9777 - val_ppl: 1.8723
Epoch 2/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - loss: 3.9471 - ppl: 1.8843 - val_loss: 3.8033 - val_ppl: 1.8052
Epoch 3/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - loss: 3.7807 - ppl: 1.8284 - val_loss: 3.7210 - val_ppl: 1.8621
Epoch 4/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - loss: 3.6731 - ppl: 1.7884 - val_loss: 3.6166 - val_ppl: 1.7432
Epoch 5/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - loss: 3.6470 - ppl: 1.7926 - val_loss: 3.6520 - val_ppl: 1.8159


In [17]:
test_loss, test_ppl = model.evaluate(test_ds)
print(f'{test_loss = }\n{test_ppl = }')

23/23 ━━━━━━━━━━━━━━━━━━━━ 20s 726ms/step - loss: 3.6122 - ppl: 1.7613
test_loss = 3.5983943939208984
test_ppl = 1.7582695484161377


In [18]:
wiki_test = lc.create_dataset_from_npy(
    npy_path= axiom_utils / 'wikitext_npy/test.npy',
    seq_len= 512,
    batch_size= 64,
    shift= 512,
    shuffle_buffer= None,
    training= False
)

In [19]:
wiki_loss, wiki_ppl = model.evaluate(wiki_test)
print(f'{wiki_loss = }\n{wiki_ppl = }')

8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 794ms/step - loss: 3.2207 - ppl: 25.1848
wiki_loss = 3.2560954093933105
wiki_ppl = 25.948017120361328


In [20]:
model.save('AxiomLM-33M-Instruct.keras')